# The Pandas library

The pandas library is used for working with data in the form of tables, time series, or datasets. It allows loading and saving data from various sources (csv, xlsx, sql databases ...)

It provides highly optimized data structures for data analysis.

We will use it for exploring data and modifying it (the ETL process).

Pandas uses the numpy library in the background, which is used for working with multidimensional data. We will look at that later.

Installation via pip: `pip install pandas` 

In [ ]:
import pandas as pd

## Loading data from CSV

In [ ]:
salary = pd.read_csv ("..\\dataset\\salary_dataset.csv")

Displaying the data

In [ ]:
salary

## Loading data from Excel
For this functionality, you need to install the openpyxl library: `pip install openpyxl`

In [ ]:
customers = pd.read_excel ("..\\dataset\\mall_customers.xlsx")

Displaying part of the data

In [ ]:
customers

## Loading data from an Sqlite3 database using an SQL query

In [ ]:
import sqlite3
cur = sqlite3.connect ("..\\dataset\\database.db")
points = pd.read_sql_query ("SELECT * FROM points", cur)
points

## Changing the data format and exporting
Sometimes it can be useful when we load structured data in one format and want to convert it to a different format.

For example, points was loaded from sqlite3 and we need to work with it in JSON.

In [ ]:
points.to_json()

Or we save it to a csv file.

In [ ]:
points.to_csv("..\\dataset\\database.csv")

If we want it in XML, we need to have the lxml library installed: `pip install lxml`

In [ ]:
points.to_xml()

## Structure of the loaded data
Pandas returns loaded data as the **DataFrame** data type.

A DataFrame is the main pandas object; it resembles a table in Excel or an SQL table.
* Each column can have a different data type.
* Each row has an index, which can be customized.

A DataFrame consists of
- **Series** - a column of data 
- **columns** - a list of columns of type Index. Column names can be edited
- **index** - the rows of the table

In [ ]:
type(points)

The structure of the dataset can be printed using `info`.

In [ ]:
points.info()


`shape` returns the dimensions of the table (number of rows, number of columns). 

In [ ]:
print(points.shape)   # (rows, columns)

`dtypes` shows the data type of each column — important for detecting incorrectly loaded columns (e.g. a number loaded as text).

In [ ]:
print(points.dtypes)

### Series
Values are stored using the numpy library. We will use that later.

We access a column via its name, and we can see that it is of type Series.

In [ ]:
points["NAME"]

You can also select subsets of columns. We define the subset as a list [].

In [ ]:
points[["NAME", "CATEGORY"]]

In [ ]:
type(points["NAME"])

### Columns
**columns** is a DataFrame attribute that contains the list of names of all the columns.

In [ ]:
points.columns

In [ ]:
type(points.columns)

Columns can be renamed. Renaming is done by modifying an entry in a dictionary.
* inplace = False, the column is renamed within the output
* inplace = True, renaming within the dataset instance

In [ ]:
points.rename(columns={"CATEGORY":"Category"})

In [ ]:
points.columns

So if we want to rename a column permanently, we need to use inplace=True

In [ ]:
points.rename(columns={"CATEGORY":"Category"}, inplace=True)

In [ ]:
points.columns

### Index
In pandas, **Index** is the base object that represents the index of rows or columns in a DataFrame or Series.

Every DataFrame has a row index (**df.index**) and a column index (**df.columns**)

Both are instances of the pandas.core.indexes.base.Index class (or its subclasses, e.g. RangeIndex)

In [ ]:
points.index

In [ ]:
points.index

In [ ]:
points.columns

The data is indexed by the internal index starting from 0.

In [ ]:
points

If we don't want to use the internally generated index, but want to use some column as the index, this can be set.

In [ ]:
points.set_index("ID", inplace=True)

The data is indexed by the ID column starting from 1. 

In [ ]:
points

In [ ]:
points.index

Sometimes it is necessary to use a date column as the index. Then you can select data by a given time span (year, quarter, ...).

In [ ]:
points.set_index("DATE", inplace=True)

In [ ]:
points

DATE was set as the Index.

In [ ]:
points.index

For time functions it needs to be set as DatetimeIndex

In [ ]:
points.index = pd.to_datetime(points.index)

In [ ]:
points.index

With a time index, we can select records from a certain range.

`sort_index()` sorts the records by index — this is necessary because selecting a range using `loc` requires sorted data.

In [ ]:

points.sort_index().loc["2020-01-01" : "2020-12-31"]

## Data samples
If the dataset is very large, displaying it can be confusing. Pandas has functions for displaying part of the data.
* head - the first records
* tail - the last records
* sample - random records

In [ ]:
points.head(5)

In [ ]:
points.tail(3)

In [ ]:
points.sample(5)

## Accessing data

### Iterating over columns
The `items()` method returns, for each column, a pair `(column_name, Series)` — analogous to `dict.items()`.

In [ ]:
for key, values in points.items():
    print (key, values)

### Selecting data by columns

In [ ]:
points[["NAME", "POINTS"]]

### Selecting a row by index
The loc method is used for this.

In [ ]:
points.loc['2020-01-10']

In [ ]:
points.loc['2021']

### Selecting a row by numeric index (position)
**iloc** is used to select a row by its position in the dataset.
* Positions are indexed from 0
* Negative numbers index from the end
* You can select just a certain range
* You can select every x-th one

In [ ]:
points.iloc[0]

In [ ]:
points.iloc[-1]

In [ ]:
points.iloc[2:4]

Every 3rd record

In [ ]:
points.iloc[::3]

### Combining column and row selection
* With loc you specify columns and index values
* With iloc you specify row indexes and column indexes

In [ ]:
points.loc["2021", "POINTS"]

In [ ]:
sum(points.loc["2021", "POINTS"])

In [ ]:
points.iloc[0, 0:3]

### Selecting by condition
A condition returns True and False values for the given indexes, depending on whether the record satisfies the condition.

In [ ]:
points["Category"] == 2

The result of the condition is then used to select rows.

In [ ]:
points[points["Category"] == 2]

Conditions can be more complex. The character | is used for the logical OR expression

In [ ]:
points[(points["Category"] == 2) | (points["POINTS"] > 8)]

The character & is used for the logical AND expression

In [ ]:
points[(points["Category"] == 2) & (points["POINTS"] > 8)]

## Applying a function to data
In some cases, the underlying data must be processed; a function must be applied to it that returns a new value.

For example, a function grades test point results with a letter grade.

In [ ]:
import numpy as np

def grade(score):
    if np.isnan(score): return np.nan
    elif 0 < score < 8:   return "D"
    elif 8 <= score <= 12: return "C"
    elif 12 <= score <= 16: return "B"
    else: return "A"

In [ ]:
points

In [ ]:
points["grade"]=points["POINTS"].apply(grade)

In [ ]:
points

## Basic statistics
After loading data from a file, you often want to get an idea of the data based on basic statistics (minimum, maximum, count, standard deviation, etc.)

The `describe` function is used for this.

In [ ]:
points.describe()

The pandas library has many statistical functions that you can run on selected data.

In [ ]:
print (points["POINTS"].mean())

In [ ]:
print (points["POINTS"].min())

## Manually creating a DataFrame

A DataFrame can be created directly from a dictionary or a list of dictionaries — this is useful for quick testing or creating small datasets.

In [ ]:
df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Carol"],
    "age":   [25, 30, 22],
    "score": [88.5, 72.0, 95.0],
})
df

## value_counts()

A quick overview of the frequency of values in a categorical column — how many times each value occurs.

In [ ]:
points["Category"].value_counts()

## sort_values()

Sorting the DataFrame by the values of a selected column. The `ascending=False` parameter sorts in descending order.

In [ ]:
points.sort_values("POINTS", ascending=False).head(5)

## groupby()

`groupby()` splits the data into groups based on the values of a column and allows you to apply an aggregate function to each group (sum, mean, count, ...).

It is the pandas equivalent of the SQL `GROUP BY` statement.

Average number of points for each category

In [ ]:
points.groupby("Category")["POINTS"].mean()

Multiple aggregations at once using agg()

In [ ]:
points.groupby("Category")["POINTS"].agg(["mean", "min", "max", "count"])

## NaN values
Just as NULL values occur in databases, a dataset can also contain NaN, i.e. an unknown value.

Sometimes it is a good idea to remove incomplete records so that they don't degrade the statistics or get involved in the learning process.

The `dropna` function deletes incomplete records.

In [ ]:
points2=points.dropna(inplace=False)
points2

Sometimes we don't want to delete incomplete records, but instead want to fill in the missing values with some value, for example the average value in the column.

In [ ]:
points["POINTS"].fillna(points["POINTS"].mean(), inplace=True)

In [ ]:
points

Filling in is not always appropriate. Should a student who didn't take the test really get an average grade of C?

In [ ]:
points["grade"]=points["POINTS"].apply(grade)

In [ ]:
points